# LLMs for Business Analytics

Starter notebook for exploring LLMs with Python.

In [1]:
import os

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pathlib import Path

DATA_DIR = Path("datasets")
df = pd.read_csv(DATA_DIR / "ESGenius_w_ref_1136q.csv")
df.head()

,query_id,new_id,query,answer,A,B,C,D,Z,ref_page,ref_doc,source_text
0,831,1,"According to the IPCC AR6 Synthesis Report, wh...",A,Implementing targeted management strategies fo...,Failing to rebuild overexploited fisheries whi...,Limiting global warming but neglecting land re...,Prioritizing disaster risk management and earl...,Not sure,46,IPCC AR6 SYR.pdf,30 Summary for Policymakers Summary for Policy...
1,889,2,According to Climate Change 2023 — Synthesis R...,B,The geographical patterns of climatic impacts ...,Geographical patterns of climatic impacts for ...,The timing of reaching a specific GWL determin...,Global warming levels are defined exclusively ...,Not sure,80-81,IPCC AR6 SYR.pdf,64 Section 2 Section 1Section 2Global Warming ...
2,892,3,Which statement accurately reflects the relati...,C,High or very high confidence in attribution is...,The text explicitly states that medium confide...,Attribution of observed physical climate chang...,Increases in heavy precipitation and sea level...,Not sure,23,IPCC AR6 SYR.pdf,7 Summary for PolicymakersSummary for Policyma...
3,983,4,According to the Climate Change 2023 — Synthes...,D,Europe,Small Islands,North America,Asia,Not sure,92,IPCC AR6 SYR.pdf,76 Section 3 Section 1Section 3011.5234 011.52...
4,1077,5,According to the Climate Change 2023 — Synthes...,A,"Early warning systems, flood-proofing of build...","Afforestation, reforestation, and levees.","Agroecological practices, disaster risk manage...","Heat Health Action Plans, vector-borne disease...",Not sure,72,IPCC AR6 SYR.pdf,"56 Section 2 Section 1Section 2wetlands, range..."


## Perturbation Engine

Generates corrupted variants of both datasets to measure LLM robustness.  
Ground truth (`answer` / `Answer`) is **never modified** — only the input context is perturbed.

| Dataset | Perturbation types | Levels | Items |
|---|---|---|---|
| ESGenius | numeric, missing, label, format, char, token | 1–3 | 1 136 × 19 = 21 584 |
| Climate Finance Bench | numeric, missing, entity, format, char, token | 1–3 | 330 × 19 = 6 270 |

All 6 perturbation types are controlled by the same two parameters, looked up from `LEVEL_TO_PARAMS`:

| Level | `alpha` | `confidence` | Meaning |
|---|---|---|---|
| 1 (mild) | 0.10 | 0.90 | 90 % of rows/elements left unchanged; 10 % magnitude when corruption occurs |
| 2 (moderate) | 0.25 | 0.75 | 75 % unchanged; 25 % magnitude |
| 3 (severe) | 0.50 | 0.50 | 50 % unchanged; 50 % magnitude |

- **`confidence`** — row-level gate: each row is left completely untouched with this probability. For `numeric` (inline numbers), the gate is applied per number rather than per row.
- **`alpha`** — magnitude of corruption when it does occur: fraction of chars/words to corrupt (`char`, `token`), fraction of sentences to drop (`missing`), jitter range for numbers (`numeric`), and the swap aggressiveness level for discrete types (`label`, `format`, `entity`).

In [2]:
import re
import numpy as np
import pandas as pd
import nltk
from pathlib import Path

nltk.download('punkt_tab', quiet=True)
np.random.seed(42)

DATA_DIR = Path("datasets")

In [3]:
import sys
sys.path.insert(0, str(Path.cwd()))

from lib.perturbation.text.sentence_pertubators import (
    perturb_text_sentence_missing_column,
    perturb_text_sentence_format_column,
    perturb_text_inline_number_column,
)
from lib.perturbation.text.character_pertubators import perturb_text_char_column
from lib.perturbation.text.token_pertubators import perturb_text_token_column
from lib.perturbation.text.entity_pertubators import (
    build_entity_pools, perturb_text_entity_column,
)
from lib.perturbation.domain.esg_pertubators import perturb_esg_confidence_column
from lib.perturbation.domain.mcq_pertubators import perturb_mcq_label_column

# Level → (alpha, confidence) used consistently by all 6 perturbation types
LEVEL_TO_PARAMS = {
    1: {"alpha": 0.10, "confidence": 0.90},
    2: {"alpha": 0.25, "confidence": 0.75},
    3: {"alpha": 0.50, "confidence": 0.50},
}

### ESGenius — MCQ perturbations

Six perturbation types applied to each of the 1 136 items:
- **numeric** — swap each confidence phrase to a different IPCC level (level 1: adjacent only; levels 2–3: any other) then multiply every number by ±10 / 25 / 50 % — via `perturb_esg_confidence_column` + `perturb_text_inline_number_column`
- **missing** — drop 10 / 25 / 50 % of sentences from `source_text` — via `perturb_text_sentence_missing_column`
- **label** — swap option texts (A/B/C/D); the answer key `answer` is never changed — via `perturb_mcq_label_column`
- **format** — shuffle sentence order → lowercase → remove punctuation from `source_text` — via `perturb_text_sentence_format_column`
- **char** — character-level noise (swap / delete / insert / replace random chars) — via `perturb_text_char_column`
- **token** — word-level noise (swap / delete / insert / replace / repeat words) — via `perturb_text_token_column`

In [4]:
esg = pd.read_csv(DATA_DIR / "ESGenius_w_ref_1136q.csv")


def _esg_numeric(df, level):
    # confidence phrase swap must run first so number jitter also touches swapped phrases
    df['source_text'] = perturb_esg_confidence_column(df=df, column='source_text', **LEVEL_TO_PARAMS[level])
    df['source_text'] = perturb_text_inline_number_column(df=df, column='source_text', **LEVEL_TO_PARAMS[level])
    return df


ESG_PERTURBATIONS = {
    "numeric": _esg_numeric,
    "missing": lambda df, level: df.assign(
        source_text=perturb_text_sentence_missing_column(df=df, column='source_text', **LEVEL_TO_PARAMS[level])
    ),
    "label": lambda df, level: perturb_mcq_label_column(
        df=df, option_columns=['A', 'B', 'C', 'D'], answer_column='answer', **LEVEL_TO_PARAMS[level]
    ),
    "format": lambda df, level: df.assign(
        source_text=perturb_text_sentence_format_column(df=df, column='source_text', **LEVEL_TO_PARAMS[level])
    ),
    "char": lambda df, level: df.assign(
        source_text=perturb_text_char_column(df=df, column='source_text', pdf='normal', **LEVEL_TO_PARAMS[level])
    ),
    "token": lambda df, level: df.assign(
        source_text=perturb_text_token_column(df=df, column='source_text', pdf='normal', **LEVEL_TO_PARAMS[level])
    ),
}

rows_esg = [esg.assign(perturbation_type='none', perturbation_level=0)]
for ptype, fn in ESG_PERTURBATIONS.items():
    for level in [1, 2, 3]:
        variant = fn(esg.copy(), level)
        variant['perturbation_type'] = ptype
        variant['perturbation_level'] = level
        rows_esg.append(variant)

esg_perturbed = pd.concat(rows_esg, ignore_index=True)
esg_perturbed.to_csv(DATA_DIR / "esgenius_perturbed.csv", index=False)

n_types = len(ESG_PERTURBATIONS)
print(f"ESGenius: {len(esg)} original × {n_types * 3 + 1} variants = {len(esg_perturbed)} rows")
esg_perturbed[['new_id', 'answer', 'perturbation_type', 'perturbation_level', 'source_text']].head(n_types * 3 + 1)

ESGenius: 1136 original × 19 variants = 21584 rows


,new_id,answer,perturbation_type,perturbation_level,source_text
0,1,A,none,0,30 Summary for Policymakers Summary for Policy...
1,2,B,none,0,64 Section 2 Section 1Section 2Global Warming ...
2,3,C,none,0,7 Summary for PolicymakersSummary for Policyma...
3,4,D,none,0,76 Section 3 Section 1Section 3011.5234 011.52...
4,5,A,none,0,"56 Section 2 Section 1Section 2wetlands, range..."
5,6,B,none,0,121 GlossaryAnnexesand in the agricultural lan...
6,7,C,none,0,121 GlossaryAnnexesand in the agricultural lan...
7,8,D,none,0,79 Long-Term Climate and Development FuturesSe...
8,9,A,none,0,viii Process The SYR was prepared in accordanc...
9,10,B,none,0,38 Section 1 Section 1This Synthesis Report (S...


### Climate Finance Bench — free-text perturbations

Six perturbation types applied to each of the 330 items:
- **numeric** — jitter all numbers in `Document extracts` by ±10 / 25 / 50 % — via `perturb_text_inline_number_column`
- **missing** — drop 10 / 25 / 50 % of sentences — via `perturb_text_sentence_missing_column`
- **entity** — swap domain-specific ESG labels using the taxonomy below — via `perturb_esg_entity_column`
- **format** — shuffle sentence order → lowercase → remove punctuation — via `perturb_text_sentence_format_column`
- **char** — character-level noise (swap / delete / insert / replace) — via `perturb_text_char_column`
- **token** — word-level noise (swap / delete / insert / replace / repeat) — via `perturb_text_token_column`

Ground truth `Answer` is never modified; scoring uses numeric extraction (PE/NR) or LLM-as-judge (LR).

#### Entity perturbation taxonomy (`lib/perturbation/domain/esg_pertubators.py`)

Seven entity types cover the labels that appear in any ESG sustainability report.
Levels control which types are active — cumulatively:

| Level | Active entity types |
|---|---|
| 1 | scope label, fiscal year |
| 2 | + emission unit, currency |
| 3 | + emission qualifier, magnitude unit, Scope 2 accounting method |

| Entity type | Example match | Pool |
|---|---|---|
| `scope_label` | "Scope 1 & 2" | Scope 1, Scope 2, Scope 3, Scope 1 & 2, Scope 1, 2 & 3 |
| `fiscal_year` | "FY2024" | any year ± 1/2/3 years |
| `emission_unit` | "MtCO2e" | tCO2e, ktCO2e, MtCO2e, GtCO2e |
| `currency` | "RMB" | RMB, USD, EUR, GBP, JPY |
| `emission_qualifier` | "net emissions" | net emissions, gross emissions |
| `magnitude_unit` | "million tons" | billion tons, million tons, thousand tons |
| `scope2_method` | "market-based" | market-based, location-based |

In [5]:
cfb = pd.read_excel(DATA_DIR / "Climate Finance Bench - Dataset.xlsx")
EXTRACT_COL = 'Document extracts'

# Build entity pools once so spaCy doesn't re-scan 330 texts for each of the 3 levels
_cfb_entity_pools = build_entity_pools(cfb[EXTRACT_COL])

CFB_PERTURBATIONS = {
    "numeric": lambda df, level: df.assign(**{
        EXTRACT_COL: perturb_text_inline_number_column(df=df, column=EXTRACT_COL, **LEVEL_TO_PARAMS[level])
    }),
    "missing": lambda df, level: df.assign(**{
        EXTRACT_COL: perturb_text_sentence_missing_column(df=df, column=EXTRACT_COL, **LEVEL_TO_PARAMS[level])
    }),
    "entity": lambda df, level: df.assign(**{
        EXTRACT_COL: perturb_text_entity_column(
            df=df, column=EXTRACT_COL, pools=_cfb_entity_pools, **LEVEL_TO_PARAMS[level]
        )
    }),
    "format": lambda df, level: df.assign(**{
        EXTRACT_COL: perturb_text_sentence_format_column(df=df, column=EXTRACT_COL, **LEVEL_TO_PARAMS[level])
    }),
    "char": lambda df, level: df.assign(**{
        EXTRACT_COL: perturb_text_char_column(df=df, column=EXTRACT_COL, pdf='normal', **LEVEL_TO_PARAMS[level])
    }),
    "token": lambda df, level: df.assign(**{
        EXTRACT_COL: perturb_text_token_column(df=df, column=EXTRACT_COL, pdf='normal', **LEVEL_TO_PARAMS[level])
    }),
}

rows_cfb = [cfb.assign(perturbation_type='none', perturbation_level=0)]
for ptype, fn in CFB_PERTURBATIONS.items():
    for level in [1, 2, 3]:
        variant = fn(cfb.copy(), level)
        variant['perturbation_type'] = ptype
        variant['perturbation_level'] = level
        rows_cfb.append(variant)

cfb_perturbed = pd.concat(rows_cfb, ignore_index=True)
cfb_perturbed.to_csv(DATA_DIR / "cfb_perturbed.csv", index=False)

n_types = len(CFB_PERTURBATIONS)
print(f"CFB: {len(cfb)} original × {n_types * 3 + 1} variants = {len(cfb_perturbed)} rows")
cfb_perturbed[['Question ID', 'Type of question', 'Answer', 'perturbation_type', 'perturbation_level']].head(n_types * 3 + 1)

CFB: 330 original × 19 variants = 6270 rows


,Question ID,Type of question,Answer,perturbation_type,perturbation_level
0,Q1,LR,According to the company's statement for fisca...,none,0
1,Q2,PE,Ali Baba Group carbon intensity dropped in FY2...,none,0
2,Q3,LR,"\nNo, the company has several projects to deca...",none,0
3,Q4,LR,"Yes, Alibaba Group operates in the logistics s...",none,0
4,Q6,PE,"Yes, Alibaba Group has identified several sign...",none,0
5,Q8,NR,"12,8 MtCO2/million RMB equivqlent to 93 200 tC...",none,0
6,Q9,LR,"Yes, Alibaba Group has several climate mitigat...",none,0
7,Q10,LR,"Yes, Alibaba discloses a Transition Plan for F...",none,0
8,Q1,LR,According to the company's statement for fisca...,none,0
9,Q2,PE,Apple's carbon intensity in the FY2023 has dec...,none,0


In [6]:
# ── Sanity checks ─────────────────────────────────────────────────────────────

for name, df_p, gt_col in [
    ("ESGenius", esg_perturbed, 'answer'),
    ("CFB",      cfb_perturbed, 'Answer'),
]:
    print(f"\n{'─'*50}")
    print(f"{name}  ({len(df_p)} rows)")
    print(df_p.groupby(['perturbation_type', 'perturbation_level']).size()
              .rename('count').to_string())

    # Ground truth must be identical to the unperturbed rows
    original_gt = df_p.loc[df_p['perturbation_level'] == 0, gt_col].reset_index(drop=True)
    for p_type in df_p['perturbation_type'].unique():
        if p_type == 'none':
            continue
        for level in [1, 2, 3]:
            subset_gt = (df_p
                         .loc[(df_p['perturbation_type'] == p_type) &
                              (df_p['perturbation_level'] == level), gt_col]
                         .reset_index(drop=True))
            assert original_gt.equals(subset_gt), \
                f"Ground truth mismatch in {name} {p_type} level {level}!"

print("\nAll ground-truth columns are intact across every perturbation variant.")


──────────────────────────────────────────────────
ESGenius  (21584 rows)
perturbation_type  perturbation_level
char               1                     1136
                   2                     1136
                   3                     1136
format             1                     1136
                   2                     1136
                   3                     1136
label              1                     1136
                   2                     1136
                   3                     1136
missing            1                     1136
                   2                     1136
                   3                     1136
none               0                     1136
numeric            1                     1136
                   2                     1136
                   3                     1136
token              1                     1136
                   2                     1136
                   3                     1136

────────────